# 13. Pipeline End-to-End Test

Runs the full pipeline ①–⑩ using `run_pipeline()` and `configs/pipeline_default.yaml`.

Pipeline order:

```
① Validation → ② Annotation → ③ Exercise Definition → ④ Preprocessing
→ ⑤ Normalization → ⑥ Phase Segmentation → ⑦ Motion Attribution
→ ⑧ Feature Extraction → ⑨ Biomech Proxy → ⑩ Biomarker Derivation
```

This notebook is the integration smoke test. It does not re-verify
each module in detail — individual step tests are in notebooks 02–12.

This notebook assumes that all previous notebooks (00–12) are passing.

Reference: `docs/code_revision_plan.md` §1 (full pipeline status)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json
from pathlib import Path

from movement.annotation import load_annotation_csv
from movement.config import LANDMARKS
from movement.io import load_pose_csv
from movement.pipeline import (
    AnnotationConfig, BiomechConfig, BiomarkerConfig, ExerciseDefinitionConfig,
    FeaturesConfig, MotionAttributionConfig, NormalizationConfig,
    PhaseSegmentationConfig, PipelineConfig, PreprocessingConfig,
    ValidationConfig, load_pipeline_config, run_pipeline,
)

print('imports OK')

## Config and Data

In [ ]:
csv_path = '../data/pose/sample/mediapipe_squat_synthetic.csv'
ann_path = '../data/pose/sample/mediapipe_squat_synthetic_annotation.csv'
def_dir  = '../data/definitions/exercises'

df_raw = load_pose_csv(csv_path)
ann_df = load_annotation_csv(ann_path)

print(f'pose data : {df_raw.shape[0]} frames, {df_raw.shape[1]} columns')
print(f'annotation: {len(ann_df)} rows')

## Build Pipeline Config — All Implemented Steps Enabled

In [ ]:
cfg = PipelineConfig()
cfg.validation          = ValidationConfig(enabled=True)
cfg.annotation          = AnnotationConfig(enabled=True, path=ann_path)
cfg.exercise_definition = ExerciseDefinitionConfig(
                              enabled=True, definitions_dir=def_dir, exercise_id='squat')
cfg.preprocessing       = PreprocessingConfig(enabled=True)
cfg.normalization       = NormalizationConfig(enabled=True)
cfg.phase_segmentation  = PhaseSegmentationConfig(enabled=True, fps_default=30.0)
cfg.motion_attribution  = MotionAttributionConfig(enabled=True)
cfg.features            = FeaturesConfig(enabled=True)
cfg.biomech             = BiomechConfig(enabled=True)
cfg.biomarker           = BiomarkerConfig(enabled=True)

print('config built — all steps ①–⑩ enabled')

## Run Pipeline

In [ ]:
result_df, report = run_pipeline(df_raw, config=cfg, landmarks=LANDMARKS)

print(f'output shape : {result_df.shape[0]} frames, {result_df.shape[1]} columns')
print(f'steps in report: {list(report.keys())}')

## Check 1: All Expected Steps Present in Report

In [ ]:
expected_steps = [
    'validation', 'annotation', 'exercise_definition', 'preprocessing',
    'normalization', 'phase_segmentation', 'motion_attribution',
    'features', 'biomech', 'biomarker', 'biomarker_scores',
]
for step in expected_steps:
    assert step in report, f'step missing from report: {step}'
print(f'PASS: all {len(expected_steps)} expected report keys present')

## Check 2: Validation Passed

In [ ]:
assert report['validation']['passed'] is True
print(f'PASS: ① validation passed')

## Check 3: Phase Labels in Output

In [ ]:
assert 'phase' in result_df.columns
rep_mask  = result_df['segment_type'] == 'rep'
n_labeled = result_df.loc[rep_mask, 'phase'].notna().sum()
assert n_labeled > 0, 'phase column empty for rep frames'
labels    = result_df.loc[rep_mask, 'phase'].dropna().unique().tolist()
print(f'PASS: ⑥ phase column — {n_labeled} frames labeled with {labels}')

## Check 4: Feature Records

In [ ]:
feat = report['features']
assert len(feat) > 0
n_rep   = sum(1 for r in feat if r.get('phase') is None)
n_phase = sum(1 for r in feat if r.get('phase') is not None)
print(f'PASS: ⑧ features — {len(feat)} records ({n_rep} rep-level, {n_phase} phase-level)')

## Check 5: Biomech Records

In [ ]:
bio = report['biomech']
assert len(bio) > 0
print(f'PASS: ⑨ biomech — {len(bio)} records')
for r in bio[:3]:
    print(f'  {r["metric_id"]:40s}  rep={r["rep_id"]}  {r["value"]:.4f} {r["unit"]}')

## Check 6: Biomarker Scores

In [ ]:
scores = report['biomarker_scores']
assert len(scores) > 0
for s in scores:
    assert 0.0 <= s['final_score'] <= 100.0
print(f'PASS: ⑩ biomarker_scores — {len(scores)} rep(s)')
for s in scores:
    print(f'  rep={s["rep_id"]}  final_score={s["final_score"]:.2f}  '
          f'spatial={s["spatial_score"]:.2f}  temporal={s["temporal_score"]:.2f}  '
          f'control={s["control_score"]:.2f}  biomech={s["biomech_score"]:.2f}')

## Check 7: Provenance — No source_fields Empty

In [ ]:
for r in report['features']:
    assert r['source_fields'], f'empty source_fields: {r["feature_id"]}'
for r in report['biomech']:
    assert r['source_fields'], f'empty source_fields: {r["metric_id"]}'
for r in report['biomarker']:
    assert r.get('source_fields'), f'empty source_fields: {r.get("biomarker_id")}'
print('PASS: provenance — source_fields non-empty for all feature / biomech / biomarker records')

## Pipeline Configuration from YAML (smoke test)

In [ ]:
# Verify that load_pipeline_config() parses the default YAML without error
yaml_cfg = load_pipeline_config('../configs/pipeline_default.yaml')
print(f'YAML config loaded:')
print(f'  validation.enabled         : {yaml_cfg.validation.enabled}')
print(f'  normalization.enabled      : {yaml_cfg.normalization.enabled}')
print(f'  phase_segmentation.enabled : {yaml_cfg.phase_segmentation.enabled}')
print(f'  features.enabled           : {yaml_cfg.features.enabled}')
print(f'  biomarker.enabled          : {yaml_cfg.biomarker.enabled}')

## Summary

| Step | Status |
|---|---|
| ① Validation | ✅ |
| ② Annotation | ✅ |
| ③ Exercise Definition | ✅ |
| ④ Preprocessing | ✅ |
| ⑤ Normalization | ✅ |
| ⑥ Phase Segmentation | ✅ |
| ⑦ Motion Attribution | ✅ |
| ⑧ Feature Extraction | ✅ |
| ⑨ Biomech Proxy | ✅ |
| ⑩ Biomarker Derivation | ✅ |

**Next steps:** Task A (Robustness Experiment Runner, notebook 14) and
Task B (Visualization charts, notebook pending) — see `docs/code_revision_plan.md` §2.